# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RAJkumar5736602/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit:** one page. **Window:** 90-day performance aggregates plus two 30-day trend windows (last 30d, days 31-60). Exact calendar dates are not in the starter CSV (anonymized) — noted as a limit.

In [1]:
import sys, pathlib
root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p/'data/raw').exists())
sys.path.insert(0, str(root/'work/scripts'))
from wk import *
import pandas as pd, numpy as np
df = load()
assert df.content_id.is_unique
print(len(df), 'rows =', df.content_id.nunique(), 'pages')

30000 rows = 30000 pages


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Features:** the 18 numeric + 3 categorical fields in `work/scripts/wk.py` (NUM, CAT). **Label:** trend_direction. **Context/grouping:** client_id. **Excluded:** see list printed below with reasons (label-derived, trend-window inputs, IDs, generation metadata).

In [2]:
for k,v in EXCLUDED.items(): print(f'{k:24s} -> {v}')
print('features:', NUM + CAT)

content_id               -> pseudonymous ID - grouping only
client_id                -> pseudonymous ID - grouping only
trend_direction          -> label source
trend_pct                -> label-derived
impressions_last_30d     -> trend input window
clicks_last_30d          -> trend input window
sessions_last_30d        -> trend input window
impressions_prev_30d     -> trend input window
clicks_prev_30d          -> trend input window
sessions_prev_30d        -> trend input window
provider_used            -> generation metadata, mostly blank
model_used               -> generation metadata, not a page property
features: ['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'impressions_90d', 'clicks_90d', 'sessions_90d', 'ai_sessions_90d', 'days_with_impressions', 'days_with_sessions', 'content_age_days', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'competition_level', 'content_type', 'main_intent']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Every claim checked by code: grain, counts per client, missing values, window consistency (last30+prev30 should not exceed 90d totals).

In [3]:
print('missing per column (non-zero only):'); m=df.isna().sum(); print(m[m>0])
print('rows per client: min/median/max', df.groupby('client_id').size().agg(['min','median','max']).tolist())
bad = (df.impressions_last_30d + df.impressions_prev_30d > df.impressions_90d + 1).sum()
print('rows where last30+prev30 > 90d impressions:', int(bad))

missing per column (non-zero only):
search_volume         2468
competition           2468
competition_level     2610
cpc                   2468
main_intent           2374
word_count            7699
char_count            7699
provider_used        21438
model_used            5733
word_count_tier       7699
char_count_tier       7699
scroll_rate            125
trend_pct             3388
dtype: int64
rows per client: min/median/max [3.0, 567.0, 7008.0]
rows where last30+prev30 > 90d impressions: 0


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Limits:** observational only; anonymized so no dates/seasonality; label is a 30-day change so noisy for low-traffic pages; 90-day totals *overlap* the trend windows (days 1-60 sit inside the 90d), so 90d features carry partial information about the label window — measured in the leakage notebook.

In [4]:
small = df.impressions_90d < 100
print('pages with <100 impressions/90d:', int(small.sum()), 'decline share there:', round(df.loc[small,'y'].mean(),3))

pages with <100 impressions/90d: 7994 decline share there: 0.389


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.